[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/user/repo/blob/main/notebook.ipynb)

In [1]:
from scipy.optimize import linprog
import numpy as np

## Vegetable Farmer Example

### 1. set up problem data

In [2]:
cost = {"lettuce": 150, "carrots": 100}    # $ per acre
price = {"lettuce": 500, "carrots": 200}   # $ per acre at full yield
yield_frac = 0.80                          # maximum yield
land = 120                                 # acres available
budget = 15000                            # $ initial investment
 
crops = ["lettuce", "carrots"]
net = np.array( [yield_frac * price[c] - cost[c] for c in crops])   # net profit per acre: [250, 60]

### 2. Constrains and bounds

In [3]:
# Constraints (A_ub @ [X, Y] <= b_ub)
A_ub = [
    [1, 1],                                  # land:   X + Y <= 120
    [cost["lettuce"], cost["carrots"]],      # budget: 150X + 100Y <= 15,000
]
b_ub = [land, budget]
 
bounds = [(0, land), (0, land)]              # 0 <= X, Y <= 120

### 3. Optimization 

In [4]:
res = linprog(c=-net, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")

In [5]:
print('optimal plan (lettuce, carrots):',res.x, ', profit',-1*res.fun)

optimal plan (lettuce, carrots): [100.   0.] , profit 25000.0


## Vegetable Farmer Insurance Example

### 1. set up problem data

In [6]:
best, worst = 25000.0, 10000.0      # net profit: no freeze / freeze
p_freeze = 0.1                    # NWS: 90% chance of no freeze
coverage = 15000.0



### 2. Fair insurance

In [7]:
### Fair insurnce
emv_ni = (1 - p_freeze) * best + p_freeze * worst  
emv_ni
z = (90./100)/(best-worst)+(best-worst)
z

15000.00006

### 3. Natural Log; Get Certainty of Equivelent, Z, and RP

In [8]:
# eu_ni = u(CE)
eu_ni = 100*(1-p_freeze)
# for neutral log
CE = np.exp((eu_ni/100)*np.log(best/worst))*worst
print('CE',CE)

CE 22811.08841388702


In [9]:
z = best-CE
print('z:', z)

z: 2188.9115861129794


In [10]:
RP = emv_ni - CE
print('RP:',RP)

RP: 688.9115861129794


### 3: An alternative way to get CE

In [11]:
from scipy.optimize import brentq

In [12]:
def u_log(x):                    # risk averse
    return 100 * np.log(x / worst) / np.log(best / worst)

In [13]:
ce = brentq(lambda x: u_log(x) - eu_ni, worst, best) # U(CE) = EU(NI); brentq(fun(x), a, b); a and b are the search range of x 

print(ce)



22811.088413887017


### 4. Hands-on - square-root 

## Hands-on: Hurricane Insurance Example - Natural log


# Bayes' probability
## Weather forecasts

In [22]:
prior = np.array([0.6, 0.3, 0.1])        # Sunny, Cloudy, Rain: P(A)
like  = np.array([0.01, 0.20, 0.90])     # forecast skill: P(FR | A)

post = like * prior / np.sum(like * prior)   # P(A | FR) = [0.038, 0.385, 0.577]

print('P[Sunny|FR], P[Cloudy|FR],P[Rain|FR]: ',post) 

P[Sunny|FR], P[Cloudy|FR],P[Rain|FR]:  [0.03846154 0.38461538 0.57692308]


## ENSO and rain fall - regression on z-score

In [27]:
"""ENSO Bayes' probability from an OLS regression on z-scores (statsmodels)."""
import pandas as pd
import statsmodels.api as sm
from scipy.stats import norm



### 1. Load data and standardization

In [28]:

ppt = pd.read_csv("cruPpt.csv", header=None, names=["year", "ppt"])
sst = pd.read_csv("Nino34.csv", header=None, names=["year", "nino34"])
df = ppt.merge(sst, on="year")

# ---- Standardize ----
df["z_rain"] = (df.ppt - df.ppt.mean()) / df.ppt.std()
df["z_sst"] = (df.nino34 - df.nino34.mean()) / df.nino34.std()


### 2. OLS: Z_rain = b0 + b1 * Z_SST 

In [29]:

X = sm.add_constant(df["z_sst"])          # adds the intercept column
model = sm.OLS(df["z_rain"], X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                 z_rain   R-squared:                       0.095
Model:                            OLS   Adj. R-squared:                  0.081
Method:                 Least Squares   F-statistic:                     6.855
Date:                Mon, 09 Nov 2026   Prob (F-statistic):             0.0110
Time:                        15:02:46   Log-Likelihood:                -91.206
No. Observations:                  67   AIC:                             186.4
Df Residuals:                      65   BIC:                             190.8
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       5.829e-16      0.117   4.98e-15      1.0

In [30]:
b1 = model.params["z_sst"]                # slope (= correlation r for z-scores)
sd = np.sqrt(1 - model.rsquared)      # residual standard error, SSR/(n-2)
print(f"\nslope b1 = {b1:.3f}, r = {np.sqrt(model.rsquared):.3f}")
print(f"residual SE = sqrt(1 - r^2) = {sd:.3f}")



slope b1 = 0.309, r = 0.309
residual SE = sqrt(1 - r^2) = 0.951


### 3. posterior tercile 

In [31]:
# normal t-statistics
t = norm.ppf(2 / 3)  
z_sst = 1.25

In [32]:
z_rain = model.predict([[1, z_sst]])[0]   # predicted mean rainfall z-score
print(z_rain)

0.38608833000094755


In [33]:
bn = norm.cdf(-t, loc=z_rain, scale=sd)
an = norm.sf(t, loc=z_rain, scale=sd)
n = 1-bn-an
print(f'BN: {100*bn:.0f}')
print(f'N:  {100*n:.0f}')
print(f'AN: {100*an:.0f}')

BN: 20
N:  32
AN: 48
